# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

Each row in the dataset represents **one webpage (one content item)** for a specific client. The row contains aggregated SEO, search performance, engagement, freshness, and content quality metrics used to evaluate whether the webpage should be prioritized for content refresh.

### Time Window

Most performance features summarize activity over the **last 90 days** (for example, impressions, clicks, sessions, users, and engagement). The dataset also includes **last 30-day** and **previous 30-day** metrics that describe recent trends and changes in performance.

This time window provides historical evidence that can be used to rank webpages for refresh without using future information.

In [10]:
import pandas as pd

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"Total Rows    : {len(df)}")
print(f"Total Columns : {df.shape[1]}")

print("\nUnique Content IDs :", df["content_id"].nunique())
print("Unique Client IDs  :", df["client_id"].nunique())

print("\nColumns containing time windows:")
time_columns = [col for col in df.columns if "90d" in col or "30d" in col]
print(time_columns)

DATASET OVERVIEW
Total Rows    : 30000
Total Columns : 44

Unique Content IDs : 30000
Unique Client IDs  : 32

Columns containing time windows:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

The following fields are used as features because they are available before making a refresh recommendation and describe the historical performance of each webpage.

- impressions_90d
- clicks_90d
- sessions_90d
- avg_position
- engagement_rate

### Label / Proxy

The label for this task is a **Refresh Priority Score (Content Opportunity Score)**. Since the dataset does not contain a direct "needs refresh" label, the score is derived from observed SEO and engagement metrics and is used to rank webpages according to their refresh priority.

### Context

The following fields provide context but are not used for prediction:

- content_id
- client_id
- content_type
- main_intent

These fields identify webpages, clients, and content categories and help interpret the results.

### Excluded

The following fields are intentionally excluded from modeling:

- provider_used
- model_used

These fields contain many missing values and describe metadata about content generation rather than webpage performance. They are not reliable predictors of content refresh priority.

In [11]:
print("=" * 50)
print("FIELD CLASSIFICATION")
print("=" * 50)

features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "engagement_rate"
]

context = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent"
]

excluded = [
    "provider_used",
    "model_used"
]

print("\nFeatures")
print(df[features].head())

print("\nContext")
print(df[context].head())

print("\nMissing values in excluded fields")
print(df[excluded].isnull().sum())

FIELD CLASSIFICATION

Features
   impressions_90d  clicks_90d  sessions_90d  avg_position  engagement_rate
0             3803          29            17          10.6             5.88
1            15320           7             9          20.3             0.00
2            12581          11            11          36.5             0.00
3            11751          58            78           6.2             1.28
4            19140          24           145          44.0             0.00

Context
             content_id          client_id     content_type    main_intent
0  content_304f48230142  client_f369cb89fc  keyword article  transactional
1  content_a1fb4e703a9e  client_4e07408562  keyword article  informational
2  content_9aa793d4d895  client_7f2253d7e2  keyword article  informational
3  content_331d6c4de07b  client_19581e27de  keyword article     commercial
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article  informational

Missing values in excluded fields
provider_used    21

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
print("=" * 50)
print("QUERY 1 - VERIFY GRAIN")
print("=" * 50)

duplicates = df["content_id"].duplicated().sum()

print("Duplicate content_id values :", duplicates)

if duplicates == 0:
    print("PASS: Every row represents one unique webpage.")
else:
    print("FAIL: Duplicate webpages found.")

QUERY 1 - VERIFY GRAIN
Duplicate content_id values : 0
PASS: Every row represents one unique webpage.


In [13]:
print("=" * 50)
print("QUERY 2 - DATASET COUNTS")
print("=" * 50)

print("Total Rows :", len(df))
print("Total Columns :", df.shape[1])
print("Unique Clients :", df["client_id"].nunique())
print("Unique Content Items :", df["content_id"].nunique())

QUERY 2 - DATASET COUNTS
Total Rows : 30000
Total Columns : 44
Unique Clients : 32
Unique Content Items : 30000


In [14]:
missing_percent = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

missing_percent = missing_percent[missing_percent > 0]

display(missing_percent.to_frame("Missing %"))

,Missing %
provider_used,71.460000
word_count,25.663333
char_count,25.663333
word_count_tier,25.663333
char_count_tier,25.663333
model_used,19.110000
trend_pct,11.293333
competition_level,8.700000
search_volume,8.226667
cpc,8.226667


### Verification Summary

The verification queries confirm that:

- Every row represents one unique webpage because there are no duplicate `content_id` values.
- The dataset contains **30,000 webpages**, **44 columns**, and **32 clients**, matching the expected structure.
- Several fields contain missing values (for example, `provider_used`, `model_used`, `word_count`, and `char_count`), so missing data must be considered during feature selection and preprocessing.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

This dataset has several important limitations that should be considered before building a machine learning model.

- The performance metrics are aggregated over fixed historical windows (90-day and 30-day periods), so they do not represent the complete lifetime performance of each webpage.
- Some fields contain missing values (such as `provider_used`, `model_used`, `word_count`, and `char_count`), which may reduce the usefulness of those features.
- The dataset is observational and describes historical search and engagement behaviour. It cannot explain the true cause of changes in traffic or rankings.
- The data should only be used with information available before making a refresh recommendation. Future information must not be used because it would introduce data leakage and produce unrealistic model performance.

In [15]:
print("=" * 50)
print("DATA LIMITS")
print("=" * 50)

missing = df.isnull().sum()
missing = missing[missing > 0]

print("Columns with missing values:")
display(missing)

print("\nNumber of columns containing missing values:")
print(len(missing))

DATA LIMITS
Columns with missing values:


search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64


Number of columns containing missing values:
13


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.